# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SS42024/shailesh-flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [5]:
# One row = one page in one calendar month which is March 2026. Each Row Summarizes how that page performed in the search of that Mont. I Chose a mid panel month so that June 2026, the final month, stays sealed as a Test Month.

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

rng = np.random.default_rng(0)
df = pd.DataFrame({
    "clicks_last_month": rng.poisson(20, 2000),
    "position": rng.uniform(1, 30, 2000),      # was "positions"
})
df["label"] = (df["clicks_last_month"] + rng.normal(0, 15, 2000) > 25).astype(int)

def score(features):
    X_tr, X_te, y_tr, y_te = train_test_split(
        df[features], df["label"], test_size=0.3, random_state=0)
    m = RandomForestClassifier(random_state=0).fit(X_tr, y_tr)
    return roc_auc_score(y_te, m.predict_proba(X_te)[:, 1])

print("honest:", score(["clicks_last_month", "position"]))

df["leak"] = df["label"] + rng.normal(0, 0.05, 2000)
print("with leak:", score(["clicks_last_month", "position", "leak"]))

honest: 0.5586962667945078
with leak: 1.0


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [ ]:
!pip -q install duckdb huggingface_hub
import duckdb
from getpass import getpass
from huggingface_hub import HfApi

# 1) Get the token: Colab Secret first, hidden prompt as a fallback
def get_token():
    try:
        from google.colab import userdata
        return userdata.get("HF_TOKEN")
    except Exception as e:
        print(f"Secret 'HF_TOKEN' not usable ({type(e).__name__}). "
              "Add it via the key icon and turn on Notebook access. "
              "For now, paste it below (hidden, not saved).")
        return getpass("HF READ token: ")

token = get_token()

# 2) Check the token itself is valid
try:
    print("Token OK for user:", HfApi().whoami(token=token)["name"])
except Exception:
    raise SystemExit("That token was rejected. Create a new plain READ token in HF settings.")

# 3) Connect DuckDB, then drop the token from memory
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf_secret (TYPE huggingface, TOKEN '{token}')")
del token

# 4) Find a file path that works (tries the partition folder, then a wildcard)
REL = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance"

def month_src(m):
    options = [
        f"read_parquet('{REL}/month={m}/*.parquet')",
        f"(SELECT * FROM read_parquet('{REL}/**/*.parquet', hive_partitioning=true) WHERE month = '{m}')",
    ]
    for src in options:
        try:
            con.sql(f"SELECT 1 FROM {src} LIMIT 1").fetchall()
            return src
        except Exception as e:
            last = str(e)
    if "403" in last or "401" in last:
        raise SystemExit("Access denied: request access on the dataset page and use a plain READ token.")
    raise SystemExit(f"No working path found. Last error:\n{last}")

MARCH, APRIL = "2026-03", "2026-04"
src = month_src(MARCH)

cols = con.sql(f"DESCRIBE SELECT * FROM {src}").df()[["column_name", "column_type"]]
print(cols.to_string())

Secret 'HF_TOKEN' not usable (SecretNotFoundError). Add it via the key icon and turn on Notebook access. For now, paste it below (hidden, not saved).


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.